# StatArb Engine — Results Walkthrough

Narrative: in-sample → walk-forward → DSR → sensitivity

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

with open('../configs/baseline.yaml') as f:
    config = yaml.safe_load(f)

## 1. Load data

In [ ]:
from statarb.data_handler import DataHandler

def flatten_universe(sectors):
    return sorted({t for ts in sectors.values() for t in ts})

universe = flatten_universe(config['universe']['sectors'])
handler = DataHandler(
    universe=universe,
    start=config['data']['start'],
    end=config['data']['end'],
    cache_dir=config['data']['cache_dir'],
)
data = handler.load()
print(f'Loaded {len(universe)} tickers, {len(handler._dates)} trading days')

## 2. In-sample pair selection (first formation window only)

In [ ]:
from statarb.strategy.pair_selection import select_pairs

form_end = handler._dates[config['walk_forward']['formation_days'] - 1]
form_data = handler.get_slice(handler._dates[0], form_end)

pairs = select_pairs(form_data, config['universe']['sectors'], config)
print(f'{len(pairs)} pairs selected:')
for p in pairs:
    print(f'  {p.ticker_y}/{p.ticker_x}  sector={p.sector}  hl={p.half_life:.1f}d  adf_p={p.adf_pvalue:.3f}')

## 3. Walk-forward backtest

In [ ]:
from statarb.validation.walk_forward import WalkForwardHarness

harness = WalkForwardHarness(handler, config)
wf_result = harness.run()

equity = wf_result['stitched_equity']
print(f'OOS equity curve: {len(equity)} bars, {equity.index[0]} to {equity.index[-1]}')

## 4. Performance metrics

In [ ]:
from statarb.analytics.metrics import full_metrics
from statarb.analytics.deflated_sharpe import deflated_sharpe_ratio
from statarb.analytics.tearsheet import print_summary

bench_handler = DataHandler(
    universe=['SPY'],
    start=config['data']['start'],
    end=config['data']['end'],
    cache_dir=config['data']['cache_dir'],
)
bench_handler.load()
bench_eq = bench_handler._data.xs('SPY', level='ticker')['adj_close']
bench_aligned = bench_eq.reindex(equity.index, method='ffill')
bench_aligned = bench_aligned / bench_aligned.iloc[0] * config['portfolio']['initial_capital']

metrics = full_metrics(equity, trade_log=wf_result['stitched_trade_log'], bench_equity=bench_aligned)

returns_arr = equity.pct_change().dropna().values
dsr = deflated_sharpe_ratio(
    observed_sr=metrics['sharpe'],
    returns=returns_arr,
    n_trials=wf_result['total_trials'],
)
print_summary(metrics, dsr_result=dsr)

## 5. Tearsheet

In [ ]:
from statarb.analytics.tearsheet import plot_tearsheet

fig = plot_tearsheet(
    equity,
    bench_equity=bench_aligned,
    trade_log=wf_result['stitched_trade_log'],
    title='StatArb Walk-Forward (OOS)',
)
plt.show()

## 6. Static OLS vs Kalman hedge ratio comparison

In [ ]:
import copy

config_ols = copy.deepcopy(config)
config_ols['strategy']['hedge_ratio'] = 'ols'

harness_ols = WalkForwardHarness(handler, config_ols)
wf_ols = harness_ols.run()
equity_ols = wf_ols['stitched_equity']

fig, ax = plt.subplots(figsize=(12, 4))
normed_kf = equity / equity.iloc[0]
normed_ols = equity_ols.reindex(normed_kf.index, method='ffill') / equity_ols.iloc[0]
ax.plot(normed_kf.index, normed_kf.values, label='Kalman hedge ratio', color='steelblue')
ax.plot(normed_ols.index, normed_ols.values, label='Static OLS', color='darkorange', linestyle='--')
ax.legend()
ax.set_title('Kalman vs Static OLS hedge ratio (OOS normalised equity)')
plt.show()

## 7. Cost sensitivity

In [ ]:
from statarb.validation.sensitivity import sweep_costs, plot_sensitivity_heatmap

def run_fn(cfg):
    h = handler.create_window_handler(cfg['data']['start'], cfg['data']['end'])
    return WalkForwardHarness(h, cfg).run()

sens_df = sweep_costs(
    run_fn, config,
    commission_multipliers=[0.5, 1.0, 2.0],
    slippage_multipliers=[0.5, 1.0, 2.0, 3.0],
    borrow_multipliers=[1.0],
)

fig = plot_sensitivity_heatmap(sens_df, metric='sharpe')
plt.show()

print('Minimum Sharpe across grid:', sens_df['sharpe'].min())
print('Fraction of grid with Sharpe > 0:', (sens_df['sharpe'] > 0).mean())